# Task 2: Network Construction & Louvain Module Detection

Build co-expression network and detect modules using **Louvain** algorithm.

**Key difference from Lab 6**: Uses Louvain instead of WGCNA (TOM + Dynamic Tree Cut).

## 1. Setup

In [1]:
import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List
import numpy as np
import pandas as pd
import networkx as nx

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")


def locate_repo_root() -> Path:
    here = Path().resolve()
    for base in [here, *here.parents]:
        if (base / "data").exists():
            return base
    raise FileNotFoundError("Could not locate repository root")


REPO_ROOT = locate_repo_root()
ARTIFACTS = REPO_ROOT / "labs/07_network_viz/assignments/artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print(f"numpy {np.__version__}")
print(f"pandas {pd.__version__}")
print(f"networkx {nx.__version__}")

numpy 2.1.3
pandas 2.2.3
networkx 3.3


## 2. Configuration

In [2]:
@dataclass
class NetworkConfig:
    handle: str
    target_gene: str = "TP53"
    corr_method: str = "pearson"
    use_abs_corr: bool = True
    adj_threshold: float = 0.4  # Lowered from 0.7 to include TP53 (max corr = 0.46)
    weighted: bool = True
    louvain_resolution: float = 1.0
    export_dir: Path = None

    def __post_init__(self):
        if self.export_dir is None:
            self.export_dir = ARTIFACTS

    def describe(self) -> Dict:
        info = asdict(self)
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = NetworkConfig(handle="AndreiCod")
CONFIG.describe()

{'handle': 'AndreiCod',
 'target_gene': 'TP53',
 'corr_method': 'pearson',
 'use_abs_corr': True,
 'adj_threshold': 0.4,
 'weighted': True,
 'louvain_resolution': 1.0,
 'export_dir': '/home/rbals/git/daha-bdhb/BDHB-lab/labs/07_network_viz/assignments/artifacts'}

In [3]:
# Load preprocessed data from Task 1
preprocessed_path = ARTIFACTS / "task1_preprocessed_expression.csv"
if not preprocessed_path.exists():
    LAB6_ARTIFACTS = REPO_ROOT / "labs/06_wgcna/assignments/artifacts"
    preprocessed_path = LAB6_ARTIFACTS / "task1_expression_preprocessed.csv"
    logging.info("Using Lab 6 preprocessed data")

assert preprocessed_path.exists(), f"Data not found: {preprocessed_path}"
expr = pd.read_csv(preprocessed_path, index_col=0)
logging.info("Loaded: %d genes x %d samples", expr.shape[0], expr.shape[1])

if CONFIG.target_gene in expr.index:
    print(f"*** {CONFIG.target_gene} present in data ***")
else:
    print(f"WARNING: {CONFIG.target_gene} not found!")
expr.head()

[INFO] Loaded: 2001 genes x 181 samples


*** TP53 present in data ***


,GSM1213669,GSM1213670,GSM1213671,GSM1213672,GSM1213673,GSM1213674,GSM1213675,GSM1213676,GSM1213677,GSM1213678,...,GSM1213840,GSM1213841,GSM1213842,GSM1213843,GSM1213844,GSM1213845,GSM1213846,GSM1213847,GSM1213848,GSM1213849
Gene,,,,,,,,,,,,,,,,,,,,,
KRT6A,12.748584,7.600682,2.522234,3.447513,5.289046,12.484240,12.487501,3.166682,12.310659,7.223532,...,2.805136,13.205127,10.422477,12.365052,7.767332,5.447799,9.245193,2.882689,12.446786,10.500599
SPRR1B,9.031516,9.583801,3.511168,8.656000,3.540976,11.032316,11.894948,3.646791,10.078840,3.445245,...,3.202791,13.705600,8.037607,9.329184,10.089060,9.008952,10.952269,4.466389,9.148314,8.220998
SCGB1A1,11.270704,6.439259,11.908472,3.578830,3.733639,6.615157,6.551806,4.529208,3.324227,9.495581,...,5.133556,4.100377,9.786214,4.064680,4.383852,5.836540,11.427470,8.748188,8.060045,9.608806
RPS4Y1,11.414617,4.690488,11.788965,4.540654,10.699047,11.721166,5.859753,8.783656,4.480617,4.657880,...,11.376956,9.229483,4.401298,12.263688,10.490783,6.078063,4.740944,4.497662,4.368025,4.180693
SPINK1,5.698196,11.904706,9.032240,7.741032,7.089012,4.170886,3.323415,3.509905,3.562707,12.295183,...,4.680854,7.370971,11.072242,5.451755,9.895093,7.162684,7.385929,6.839303,6.015390,11.157566


## 3. Core Functions

In [4]:
def compute_correlation_matrix(
    df: pd.DataFrame, method: str = "pearson"
) -> pd.DataFrame:
    corr = df.T.corr(method=method)
    logging.info(
        "Computed %s correlation: %d x %d", method, corr.shape[0], corr.shape[1]
    )
    return corr


def build_adjacency_matrix(
    corr: pd.DataFrame, threshold: float, use_abs: bool = True, weighted: bool = True
) -> pd.DataFrame:
    corr_vals = corr.abs() if use_abs else corr.copy()
    if weighted:
        adj = corr_vals.where(corr_vals >= threshold, 0)
    else:
        adj = (corr_vals >= threshold).astype(int)
    np.fill_diagonal(adj.values, 0)
    n_edges = (adj.values > 0).sum() // 2
    logging.info("Adjacency matrix: %d edges (threshold=%.2f)", n_edges, threshold)
    return adj


def build_networkx_graph(adj: pd.DataFrame, weighted: bool = True) -> nx.Graph:
    G = (
        nx.from_pandas_adjacency(adj)
        if weighted
        else nx.from_pandas_adjacency((adj > 0).astype(int))
    )
    isolated = list(nx.isolates(G))
    G.remove_nodes_from(isolated)
    logging.info(
        "Graph: %d nodes, %d edges (removed %d isolated)",
        G.number_of_nodes(),
        G.number_of_edges(),
        len(isolated),
    )
    return G


def louvain_module_detection(G: nx.Graph, resolution: float = 1.0) -> Dict[str, int]:
    communities = nx.community.louvain_communities(
        G, weight="weight", resolution=resolution, seed=42
    )
    gene_to_module = {}
    for module_id, community in enumerate(communities):
        for gene in community:
            gene_to_module[gene] = module_id
    n_modules = len(communities)
    sizes = [len(c) for c in communities]
    logging.info(
        "Louvain: %d modules (min=%d, max=%d)", n_modules, min(sizes), max(sizes)
    )
    return gene_to_module

## 4. Execute Pipeline

In [5]:
print("=" * 60)
print("NETWORK CONSTRUCTION PIPELINE")
print("=" * 60)
corr_matrix = compute_correlation_matrix(expr, method=CONFIG.corr_method)
print(f"Correlation matrix: {corr_matrix.shape}")

NETWORK CONSTRUCTION PIPELINE


[INFO] Computed pearson correlation: 2001 x 2001


Correlation matrix: (2001, 2001)


In [6]:
adj_matrix = build_adjacency_matrix(
    corr_matrix,
    threshold=CONFIG.adj_threshold,
    use_abs=CONFIG.use_abs_corr,
    weighted=CONFIG.weighted,
)
if CONFIG.target_gene in adj_matrix.index:
    tp53_edges = (adj_matrix.loc[CONFIG.target_gene] > 0).sum()
    print(f"TP53 has {tp53_edges} connections")

[INFO] Adjacency matrix: 148240 edges (threshold=0.40)


TP53 has 20 connections


In [7]:
# Check TP53 correlation distribution to find appropriate threshold
tp53_corrs = corr_matrix.loc["TP53"].abs().drop("TP53").sort_values(ascending=False)
print("TP53 top 20 correlations:")
print(tp53_corrs.head(20))
print(f"\nTP53 correlation stats:")
print(f"  Max:    {tp53_corrs.max():.4f}")
print(f"  95th:   {tp53_corrs.quantile(0.95):.4f}")
print(f"  90th:   {tp53_corrs.quantile(0.90):.4f}")
print(f"  Median: {tp53_corrs.median():.4f}")
print(f"\nGenes with |corr| >= 0.5: {(tp53_corrs >= 0.5).sum()}")
print(f"Genes with |corr| >= 0.6: {(tp53_corrs >= 0.6).sum()}")
print(f"Genes with |corr| >= 0.7: {(tp53_corrs >= 0.7).sum()}")

TP53 top 20 correlations:
Gene
MBNL2 /// MBNL2       0.464712
GTF2H2B               0.460662
N4BP2L2 /// U50535    0.457925
LCOR                  0.455124
TMEM263               0.453952
HEATR5A               0.449697
NEK7                  0.446135
NRIP1                 0.441774
LYSMD3                0.436469
DNAJB14               0.431904
DKFZP586I1420         0.425137
ANKRD36B              0.424674
CHD1                  0.424383
BAZ2B                 0.412720
NOTCH2NL              0.408319
PPIP5K2               0.408124
FAM8A1                0.407526
GSAP                  0.407362
MAN2A1                0.406243
CLK1                  0.401813
Name: TP53, dtype: float64

TP53 correlation stats:
  Max:    0.4647
  95th:   0.3188
  90th:   0.2625
  Median: 0.1116

Genes with |corr| >= 0.5: 0
Genes with |corr| >= 0.6: 0
Genes with |corr| >= 0.7: 0


In [8]:
G = build_networkx_graph(adj_matrix, weighted=CONFIG.weighted)
print(
    f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}, Density: {nx.density(G):.4f}"
)
if CONFIG.target_gene in G.nodes():
    print(f"*** TP53 in network (degree={G.degree(CONFIG.target_gene)}) ***")

[INFO] Graph: 1948 nodes, 148240 edges (removed 53 isolated)


Nodes: 1948, Edges: 148240, Density: 0.0782
*** TP53 in network (degree=20) ***


In [9]:
print("\nLOUVAIN MODULE DETECTION")
gene_modules = louvain_module_detection(G, resolution=CONFIG.louvain_resolution)
modules_df = pd.DataFrame(
    [{"Gene": g, "Module": m} for g, m in gene_modules.items()]
).sort_values(["Module", "Gene"])
print(f"Total genes in modules: {len(modules_df)}")
modules_df.head(10)


LOUVAIN MODULE DETECTION


[INFO] Louvain: 11 modules (min=2, max=527)


Total genes in modules: 1948


,Gene,Module
312,A2M-AS1,0
69,ABCA3,0
368,ABCC3,0
183,ABCC6 /// ABCC6P1 /// ABCC6P2 /// LOC101930322,0
151,ACAD8,0
136,ACKR1,0
50,ACTG1P4 /// AMY1A /// AMY1B /// AMY1C /// AMY2...,0
287,ADA,0
6,ADH1B,0
35,ADIRF,0


In [10]:
module_sizes = modules_df["Module"].value_counts().sort_index()
print("Module sizes:")
for mod_id, size in module_sizes.items():
    print(f"  Module {mod_id}: {size} genes")
if CONFIG.target_gene in gene_modules:
    tp53_mod = gene_modules[CONFIG.target_gene]
    print(f"\n*** TP53 is in Module {tp53_mod} ({module_sizes[tp53_mod]} genes) ***")

Module sizes:
  Module 0: 511 genes
  Module 1: 43 genes
  Module 2: 498 genes
  Module 3: 2 genes
  Module 4: 268 genes
  Module 5: 3 genes
  Module 6: 527 genes
  Module 7: 2 genes
  Module 8: 86 genes
  Module 9: 6 genes
  Module 10: 2 genes

*** TP53 is in Module 4 (268 genes) ***


## 5. Hub Genes

In [11]:
def identify_hub_genes(
    G: nx.Graph, gene_modules: Dict[str, int], top_n: int = 5
) -> pd.DataFrame:
    degree_cent = nx.degree_centrality(G)
    betweenness_cent = nx.betweenness_centrality(G)
    hub_data = []
    for module_id in sorted(set(gene_modules.values())):
        module_genes = [g for g, m in gene_modules.items() if m == module_id]
        centralities = [
            (g, degree_cent.get(g, 0), betweenness_cent.get(g, 0)) for g in module_genes
        ]
        centralities.sort(key=lambda x: x[1], reverse=True)
        for rank, (gene, deg, bet) in enumerate(centralities[:top_n], 1):
            hub_data.append(
                {
                    "Module": module_id,
                    "Gene": gene,
                    "Rank": rank,
                    "Degree": G.degree(gene),
                    "DegreeCentrality": round(deg, 4),
                    "BetweennessCentrality": round(bet, 4),
                }
            )
    return pd.DataFrame(hub_data)


hub_genes = identify_hub_genes(G, gene_modules, top_n=5)
print(f"Identified {len(hub_genes)} hub genes")
hub_genes

Identified 44 hub genes


,Module,Gene,Rank,Degree,DegreeCentrality,BetweennessCentrality
0,0,TMEM125,1,628,0.3225,0.0049
1,0,C16orf89,2,622,0.3195,0.0051
2,0,SELENBP1,3,620,0.3184,0.0041
3,0,GPR116,4,616,0.3164,0.0041
4,0,SFTA2,5,610,0.3133,0.0042
5,1,MAGEA6,1,129,0.0663,0.0018
6,1,MAGEA3 /// MAGEA6,2,127,0.0652,0.0019
7,1,MAGEA12,3,110,0.0565,0.0021
8,1,MAGEA1,4,93,0.0478,0.0019
9,1,MAGEA2 /// MAGEA2B,5,75,0.0385,0.0010


## 6. Export Results

In [12]:
# Save deliverables
modules_path = CONFIG.export_dir / "task2_modules.csv"
modules_df.to_csv(modules_path, index=False)
logging.info("[OK] Modules saved: %s", modules_path)

hubs_path = CONFIG.export_dir / "task2_hub_genes.csv"
hub_genes.to_csv(hubs_path, index=False)
logging.info("[OK] Hub genes saved: %s", hubs_path)

corr_path = CONFIG.export_dir / "task2_correlation_matrix.csv"
corr_matrix.to_csv(corr_path)
logging.info("[OK] Correlation matrix saved")

nx.set_node_attributes(G, gene_modules, "module")
gml_path = CONFIG.export_dir / "task2_network.gml"
nx.write_gml(G, str(gml_path))
logging.info("[OK] Network saved as GML")

print(f"\n{'=' * 60}")
print("TASK 2 COMPLETE")
print(f"{'=' * 60}")
print(f"Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Modules: {len(set(gene_modules.values()))}")
if CONFIG.target_gene in gene_modules:
    print(f"TP53 module: {gene_modules[CONFIG.target_gene]}")
print(f"\nDeliverables: task2_modules.csv, task2_hub_genes.csv")

[INFO] [OK] Modules saved: /home/rbals/git/daha-bdhb/BDHB-lab/labs/07_network_viz/assignments/artifacts/task2_modules.csv
[INFO] [OK] Hub genes saved: /home/rbals/git/daha-bdhb/BDHB-lab/labs/07_network_viz/assignments/artifacts/task2_hub_genes.csv
[INFO] [OK] Correlation matrix saved
[INFO] [OK] Network saved as GML



TASK 2 COMPLETE
Network: 1948 nodes, 148240 edges
Modules: 11
TP53 module: 4

Deliverables: task2_modules.csv, task2_hub_genes.csv
